<a href="https://colab.research.google.com/github/giovannicarnevali/distribution_italy/blob/main/FTSEMIB_step1_candele.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trading System FTSE MIB — STEP 1: dati e grafici a candele
**Repo:** `github.com/giovannicarnevali/trading` · notebook per Google Colab

Cosa fa questo notebook:
1. scarica i dati giornalieri OHLCV delle componenti del FTSE MIB (API Yahoo Finance) + l'indice;
2. li salva in CSV nella cartella `data/` (riutilizzabili);
3. disegna il **grafico pivot**: candele giapponesi + volumi, con **menu a tendina** per cambiare azione e scegliere il periodo.

> **Come salvarlo su GitHub**: in Colab, menu `File → Salva una copia su GitHub` → scegli il repo `giovannicarnevali/trading`. Fatto: il notebook vive nel repo e lo riapri da lì con un click.

Allo STEP 2 su questi stessi grafici sovrapporremo le regole del trading system (filtro di regime MA10 mensile, momentum 6m skip-1, segnali di ingresso/uscita della rotazione).

In [4]:
#@title 1) Configurazione — componenti FTSE MIB { display-mode: "form" }
# Lista corrente delle componenti (ticker Yahoo, suffisso .MI = Borsa Italiana).
# Nota: l'API di Borsa Italiana non espone dati OHLC gratuiti via API pubblica;
# Yahoo e' la fonte pratica. La pagina 'components' di Yahoo non da' lo storico:
# si scarica titolo per titolo con l'endpoint 'chart'.
TICKERS = [
 'A2A.MI','AMP.MI','AZM.MI','BAMI.MI','BC.MI','BGN.MI','BMED.MI','BMPS.MI',
 'BPE.MI','BZU.MI','CPR.MI','DIA.MI','ENEL.MI','ENI.MI','ERG.MI','FBK.MI',
 'G.MI','HER.MI','IG.MI','INW.MI','IP.MI','ISP.MI','IVG.MI','LDO.MI','MB.MI',
 'MONC.MI','NEXI.MI','PIRC.MI','PRY.MI','PST.MI','RACE.MI','REC.MI','SPM.MI',
 'SRG.MI','STLAM.MI','STMMI.MI','TEN.MI','TIT.MI','TRN.MI','UCG.MI','UNI.MI',
]
INDICE = 'FTSEMIB.MI'
print(f'{len(TICKERS)} componenti + indice')

41 componenti + indice


In [5]:
#@title 2) Download dei dati (Yahoo chart API) e salvataggio CSV
import urllib.request, json, os, time
import pandas as pd

def fetch(sym):
    """Scarica l'intero storico giornaliero OHLCV di un ticker da Yahoo.
    IMPORTANTE: senza period1/period2 espliciti Yahoo restituisce dati MENSILI."""
    url = (f'https://query1.finance.yahoo.com/v8/finance/chart/{sym}'
           '?period1=0&period2=9999999999&interval=1d')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    j = json.loads(urllib.request.urlopen(req, timeout=30).read())
    r = j['chart']['result'][0]; q = r['indicators']['quote'][0]
    df = pd.DataFrame({'ts': r['timestamp'], 'open': q['open'], 'high': q['high'],
                       'low': q['low'], 'close': q['close'], 'volume': q['volume']})
    adj = r['indicators'].get('adjclose', [{}])[0].get('adjclose')
    if adj: df['adjclose'] = adj
    df['date'] = pd.to_datetime(df['ts'], unit='s').dt.normalize()
    df = df.drop(columns='ts').dropna(subset=['close']).set_index('date')
    return df[~df.index.duplicated(keep='last')]

os.makedirs('data', exist_ok=True)
DATA = {}
for sym in TICKERS + [INDICE]:
    try:
        DATA[sym] = fetch(sym)
        DATA[sym].to_csv(f'data/{sym}.csv')
        print(f'{sym:10s} {len(DATA[sym]):6d} sedute  dal {DATA[sym].index[0].date()}')
        time.sleep(0.3)   # gentilezza verso l'API
    except Exception as e:
        print(f'{sym:10s} ERRORE: {e}')
print('\nFatto. CSV in ./data/')

A2A.MI       6865 sedute  dal 2000-01-03
AMP.MI       6410 sedute  dal 2001-10-01
AZM.MI       5688 sedute  dal 2004-07-07
BAMI.MI      6865 sedute  dal 2000-01-03
BC.MI        3693 sedute  dal 2012-04-27
BGN.MI       5077 sedute  dal 2006-11-15
BMED.MI      6865 sedute  dal 2000-01-03
BMPS.MI      7000 sedute  dal 1999-06-25
BPE.MI       8111 sedute  dal 1995-03-24
BZU.MI      10127 sedute  dal 1987-07-01
CPR.MI       6471 sedute  dal 2001-07-06
DIA.MI       4907 sedute  dal 2007-07-19
ENEL.MI      6909 sedute  dal 1999-11-02
ENI.MI       7933 sedute  dal 1995-11-29
ERG.MI       7250 sedute  dal 1998-07-10
FBK.MI       3144 sedute  dal 2014-07-02
G.MI        10370 sedute  dal 1986-07-25
HER.MI       6021 sedute  dal 2003-03-28
IG.MI        2546 sedute  dal 2016-11-07
INW.MI       2900 sedute  dal 2015-06-22
IP.MI        7659 sedute  dal 1996-12-16
ISP.MI       8110 sedute  dal 1995-03-24
IVG.MI       1238 sedute  dal 2022-01-03
LDO.MI       5420 sedute  dal 2005-07-18
MB.MI       1016

In [6]:
#@title 3) GRAFICO PIVOT — candele + volumi, menu a tendina per cambiare azione
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown, IntSlider
import pandas as pd

def grafico(titolo, anni):
    df = DATA[titolo]
    d = df[df.index >= df.index[-1] - pd.Timedelta(days=anni*365)]
    sma50  = df['close'].rolling(50).mean().reindex(d.index)
    sma200 = df['close'].rolling(200).mean().reindex(d.index)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.75, 0.25], vertical_spacing=0.03)
    fig.add_trace(go.Candlestick(x=d.index, open=d['open'], high=d['high'],
                                 low=d['low'], close=d['close'], name=titolo,
                                 increasing_line_color='#26a69a',
                                 decreasing_line_color='#ef5350'), row=1, col=1)
    fig.add_trace(go.Scatter(x=d.index, y=sma50,  name='SMA 50',
                             line=dict(width=1.2, color='#ff9800')), row=1, col=1)
    fig.add_trace(go.Scatter(x=d.index, y=sma200, name='SMA 200',
                             line=dict(width=1.2, color='#3f51b5')), row=1, col=1)
    colori = ['#26a69a' if c >= o else '#ef5350' for o, c in zip(d['open'], d['close'])]
    fig.add_trace(go.Bar(x=d.index, y=d['volume'], marker_color=colori,
                         name='Volume', showlegend=False), row=2, col=1)
    fig.update_layout(height=680, title=f'{titolo} — ultimi {anni} anni',
                      xaxis_rangeslider_visible=False,
                      legend=dict(orientation='h', y=1.02),
                      margin=dict(l=40, r=20, t=60, b=20))
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat','mon'])])  # niente buchi weekend
    fig.show()

interact(grafico,
         titolo=Dropdown(options=sorted(DATA), value='ENI.MI', description='Azione'),
         anni=IntSlider(min=1, max=25, value=3, description='Anni'));

interactive(children=(Dropdown(description='Azione', index=13, options=('A2A.MI', 'AMP.MI', 'AZM.MI', 'BAMI.MI…

In [7]:
#@title 4) (Opzionale) Esporta tutto in un file Excel — un foglio per titolo
#@markdown Candele native di Excel: seleziona i dati di un foglio → Inserisci → Grafico azionario (Apertura-Max-Min-Chiusura).
ANNI_EXCEL = 3  #@param {type:"integer"}
import pandas as pd
with pd.ExcelWriter('ftsemib_ohlcv.xlsx', engine='openpyxl') as xl:
    for sym, df in DATA.items():
        d = df[df.index >= df.index[-1] - pd.Timedelta(days=ANNI_EXCEL*365)]
        d[['open','high','low','close','volume']].to_excel(xl, sheet_name=sym.replace('.MI','')[:31])
print('Creato ftsemib_ohlcv.xlsx — scaricalo dal pannello file di Colab (icona cartella a sinistra)')

Creato ftsemib_ohlcv.xlsx — scaricalo dal pannello file di Colab (icona cartella a sinistra)


---
## Prossimo: STEP 2 — il trading system sui grafici
Su questo stesso grafico pivot aggiungeremo:
- il **filtro di regime** (indice sopra/sotto la MA10 delle chiusure mensili → sfondo verde/rosso);
- il **ranking momentum 6m skip-1** del titolo selezionato e la sua posizione in classifica;
- i **segnali della rotazione**: frecce di ingresso/uscita nei mesi in cui il TS ha comprato/venduto il titolo, con il ledger degli ordini;
- (modulo BTC: ensemble dei rilevatori + uscita MA21).

Nota dati: i ticker Yahoo coprono le componenti **attuali** — per i backtest senza survivorship bias usiamo il dataset point-in-time dai bollettini di Borsa Italiana (vedi progetto).